# Python for Poets 2026 - Workbook 4 - Speeches in Parliament

In this workbook we bring together what we did with metadata (Workbook 2) and distinctive words (Workbook 3), on a much bigger corpus: about 96,000 speeches made in the British House of Commons between 1979 and 2021.

The bigger the corpus, the easier it is to find differences, and the harder it is to say what they mean. Along the way we'll meet three problems that every large-scale comparison runs into: a column of metadata that isn't what it seems, words that look typical of a group but belong to one person, and groups that differ in more than one way at once.

## Setup

In [ ]:
#@title Install and import the `python4poets` package
try:
    import python4poets as p4p
except ImportError:
    # Not installed yet: install it into this notebook's Python, then import
    %pip install -q git+https://github.com/quadrismegistus/python-for-poets
    import python4poets as p4p

import pandas as pd
import plotnine as p9
import warnings
warnings.filterwarnings('ignore')

## The corpus

*Hansard* is the official record of what is said in Parliament, named after Thomas Curtis Hansard, its nineteenth-century printer. It is close to a transcript, but lightly edited: repetitions and slips are tidied up.

Our corpus comes from Evan Odell's dataset of every Commons speech from May 1979 to April 2021, with each speaker's party. We've added a few more columns (gender, frontbench posts) from Parliament's own database of MPs, and taken a sample:

- only speeches of 50 to 1,000 words (no one-line interjections, no three-hour budget speeches);
- each year, 1,000 speeches by Conservative MPs and 1,000 by Labour MPs, chosen at random;
- speeches by other parties (Liberal Democrats, SNP, and the rest) in proportion to how much they actually spoke that year.

A little Westminster vocabulary will help:

| term | meaning |
|---|---|
| **government** | the party (or parties) with a majority, whose leader is Prime Minister: Conservative 1979–97, Labour 1997–2010, Conservative–Liberal Democrat coalition 2010–15, Conservative since 2015 |
| **opposition** | everyone else; the largest other party is "the Opposition" |
| **frontbench** | MPs with a job: ministers in the government, and their opposite numbers ("shadow" ministers) in the opposition. They sit on the front benches, facing each other |
| **backbench** | MPs without a government or shadow post: everyone else |

In [ ]:
#@title Download and open the Hansard sample
df = p4p.load_hansard()
df

| column | meaning |
|---|---|
| `date`, `year`, `decade` | when the speech was given |
| `speaker` | who gave it |
| `gender` | the speaker's gender, as recorded by Parliament |
| `party` | the speaker's party, as recorded in Hansard |
| `party_main` | the same, grouped: Conservative, Labour, Liberal Democrat (including the Liberals and SDP before 1988), SNP, Other |
| `side` | whether the speaker's party was in government or opposition that day |
| `frontbench`, `post` | whether the speaker held a government or shadow post that day, and which |
| `constituency` | the place the MP represents |
| `major_heading`, `minor_heading` | the debate the speech was part of |
| `word_count` | the speech's length in words |
| `id`, `txt` | the speech's ID in Hansard, and the speech itself |

## Part 1: Metadata

### Who speaks?

In [ ]:
#@title Summary statistics for the quantitative columns
df.describe()

∇ *Why is the smallest `word_count` exactly 50, and the largest exactly 1000?*

In [ ]:
#@title Number of speeches per party
df['party_main'].value_counts()

∇ *Conservative and Labour have exactly the same number of speeches. Why? (Look back at how the sample was made.) What does that mean we **can't** use this sample to find out?*

In [ ]:
#@title TODO: Number of speeches per party, before grouping (`party`)
df['REPLACE_ME'].value_counts()

In [ ]:
#@title Number of speeches by gender
df['gender'].value_counts()

In [ ]:
#@title TODO: Number of speeches by `side`, and by `frontbench`
df['REPLACE_ME'].value_counts()

In [ ]:
#@title Who speaks the most?
df['speaker'].value_counts().head(20)

∇ *What do the people at the top of this list have in common? Look up any names you don't know. Is anyone there for a different reason?*

In [ ]:
#@title TODO: What are the most common debates (`major_heading`)?
df['REPLACE_ME'].value_counts().head(20)

∇ *Some debates appear under two slightly different names. Why might the same kind of debate be recorded differently in different decades?*

### Gender and party

As in Workbook 2, a contingency table counts how often each combination of two columns occurs. `p4p.crosstab_stats` also tests it for us, and gives the observed / expected table.

In [ ]:
#@title Contingency table: `gender` vs. `party_main`
pd.crosstab(df['gender'], df['party_main'])

In [ ]:
#@title Statistical significance and observed / expected
chi2, p, obs_exp = p4p.crosstab_stats(df, 'gender', 'party_main')

print(f'Chi-squared statistic: {chi2:.1f}')
print(f'P-value: {p}')
obs_exp

∇ *Which party's speeches are most often by women, compared to what we'd expect by chance? Which least?*

∇ *This table counts **speeches**, not MPs. How could the two give different answers?*

In [ ]:
#@title TODO: Contingency table and statistics: `gender` vs. `frontbench`
chi2, p, obs_exp = p4p.crosstab_stats(df, 'REPLACE_ME', 'REPLACE_ME_ALSO')

print(f'Chi-squared statistic: {chi2:.1f}')
print(f'P-value: {p}')
obs_exp

### Change over time

What share of each party's speeches were made by women, year by year? A trick: if we make a column that is `True` for women's speeches and `False` otherwise, its average (`mean`) is the share that are `True`.

In [ ]:
#@title Share of speeches by women, by year and party
df['by_woman'] = df['gender'] == 'female'

women_by_year = df.groupby(['year', 'party_main'])['by_woman'].mean().reset_index(name='share_by_women')
women_by_year

In [ ]:
#@title Plot it
fig = p9.ggplot(women_by_year, p9.aes(x='year', y='share_by_women', color='party_main'))
fig += p9.geom_point()
fig += p9.geom_smooth()
fig

∇ *What happens to Labour in 1997? (Look up "all-women shortlists".) When do the Conservatives catch up, if they do?*

∇ *Why do the points for the SNP and "Other" jump around so much more than the others?*

### Can we trust a column?

Metadata looks like plain fact, but every column was made by someone, somehow. Let's look at how many speeches were given by frontbenchers each year, on each side.

In [ ]:
#@title Share of speeches by frontbenchers, by year and side
frontbench_by_year = df.groupby(['year', 'side'])['frontbench'].mean().reset_index(name='share_frontbench')

fig = p9.ggplot(frontbench_by_year, p9.aes(x='year', y='share_frontbench', color='side'))
fig += p9.geom_point()
fig += p9.geom_smooth()
fig

∇ *On the government side, the share is fairly steady. On the opposition side, it rises from about one speech in eight to more than half. Is that a change in how Parliament works, or in the data?*

Here is how the `frontbench` column was made: for each MP, Parliament's database lists the posts they held, with start and end dates, and we checked whether the speech's date fell inside one. But the database's records of *shadow* posts are much less complete for the 1980s and 90s than for recent years. In 1979 it lists only 23 opposition MPs with a shadow post who spoke that year; by 2020, 193.

Some of the rise may be real (since the 2000s, smaller parties have named spokespeople for almost everything), but much of it is probably in the record, not in history. If we compare frontbench and backbench speeches, we should stick to the period where the record is good: since 1997.

∇ *Can you think of a column in your own corpus (or in the Novel450 or State of the Union corpora) that might have a similar problem?*

In [ ]:
#@title TODO: Plot the average `word_count` by year, colored by a column of your choice
length_by_year = df.groupby(['year', 'REPLACE_ME'])['word_count'].mean().reset_index(name='average_length')

fig = p9.ggplot(length_by_year, p9.aes(x='year', y='average_length', color='REPLACE_ME'))
fig += p9.geom_point()
fig += p9.geom_smooth()
fig

## Part 2: Distinctive words

### Conservative vs. Labour

Let's start as in Workbook 3: which words are most typical of Conservative speeches, and which of Labour ones? (Counting the words in 96,000 speeches takes a few seconds the first time; after that the counts are remembered.)

In [ ]:
#@title Distinctive words: Conservative vs. Labour
result = p4p.distinctive_words_by_group(df, 'party_main', 'Conservative', 'Labour')
p4p.plot_distinctive_words(result)

∇ *Do these look like Conservative and Labour words? Do you recognise them at all?*

The table has two columns we didn't have in Workbook 3: `texts_Conservative` and `texts_Labour`, the number of *speeches* in each party that use the word.

In [ ]:
#@title How many speeches use the top words?
result[['word', 'count_Conservative', 'count_Labour', 'texts_Conservative', 'texts_Labour', 'log_ratio', 'p_adj']].head(20)

In [ ]:
#@title Read one in context
p4p.kwic(df, 'spitalfields')

∇ *Who is using this word? Is it a Labour word, or one MP's word?*

### One speaker, many uses

The significance test in Workbook 3 imagines that every word in the corpus was drawn at random, independently of every other word, like drawing marbles from a bag. That's never quite true of language (once you've said "Spitalfields", you are much more likely to say it again), but in a small corpus of long texts it doesn't matter much.

In a large corpus of short texts, it matters a lot. An MP who campaigns on one issue can use a rare word dozens of times, in a handful of speeches, and the test counts each use as fresh evidence about their whole party.

One simple fix is to ignore words used in only a few speeches. `min_texts=20` keeps only words used in at least 20 speeches (from either party):

In [ ]:
#@title Distinctive words, used in at least 20 speeches
result = p4p.distinctive_words_by_group(df, 'party_main', 'Conservative', 'Labour', min_texts=20)
p4p.plot_distinctive_words(result)

∇ *What kinds of words are these now? Are there still words that belong to one person, one place, or one moment?*

∇ *Try `min_texts=100` in the cell above. What changes? What might we lose by setting it too high?*

### Big differences, or sure differences?

So far we've ranked words by the **log ratio**: how many *times* more often one party uses them. That favours rarer words, since a word used 30 times vs. 2 has a huge ratio. We can instead rank them by **G²**: how *sure* we are that the difference is real. That favours common words, since a small difference in a common word is a lot of evidence.

In [ ]:
#@title Distinctive words, ranked by G²
p4p.plot_distinctive_words(result, by='g2')

∇ *Why is "friend" the most Conservative word? And why might "Tory" be the most Labour word, and "Labour" one of the most Conservative?*

In [ ]:
#@title "my hon. Friend", in context (a random sample of 20)
p4p.kwic(df, 'my hon friend').sample(20, random_state=1)

In the Commons, MPs never address each other by name. They speak to the Speaker, and refer to each other by role: "the hon. Member for Bolsover", "my hon. Friend" (an MP of one's own party), "the hon. Gentleman" or "the hon. Lady" (an MP of another party), "right hon." for senior members.

So "my hon. Friend" is what you say to your own side. Who does that most? Ministers answering their own backbenchers' questions ("I thank my hon. Friend..."), and backbenchers asking friendly questions of their own ministers ("Does my right hon. Friend agree..."). In other words: the party in **government**. And the Conservatives were in government for 29 of these 42 years.

### Party, or position?

Workbook 3 asked "party, or period?". Here we can ask "party, or position?". Let's compare government and opposition directly:

In [ ]:
#@title Distinctive words: government vs. opposition
result_side = p4p.distinctive_words_by_group(df, 'side', 'government', 'opposition', min_texts=20)
p4p.plot_distinctive_words(result_side, by='g2')

∇ *How many of the "Conservative" words from before are here too? What does the language of government sound like, and the language of opposition?*

To see what's left of *party*, we can hold position constant: compare Conservatives and Labour only when they were in government.

In [ ]:
#@title Conservative vs. Labour, only in government
df_government = df[df['side'] == 'government']

result_government = p4p.distinctive_words_by_group(df_government, 'party_main', 'Conservative', 'Labour', min_texts=20)
p4p.plot_distinctive_words(result_government)

∇ *What are these words about? Hint: when was Labour in government?*

Holding position constant brought period back in. In government, Labour only ever means 1997–2010, so Labour words are 1997–2010 words. In this corpus, party, position and period are tangled together, and no single comparison can pull them all apart.

What we can do is make several comparisons, each with a different weakness, and see which words survive all of them.

In [ ]:
#@title TODO: Conservative vs. Labour, only in opposition
df_opposition = df[df['side'] == 'REPLACE_ME']

result_opposition = p4p.distinctive_words_by_group(df_opposition, 'party_main', 'Conservative', 'Labour', min_texts=20)
p4p.plot_distinctive_words(result_opposition, by='g2')

∇ *Which words are distinctive of Labour (or of the Conservatives) in all three comparisons: all speeches, government only, and opposition only? Use the tables (`result`, `result_government`, `result_opposition`) to check. Are those more convincing "party" words? Why?*

In [ ]:
#@title TODO: Look up one word in all three tables
word = 'REPLACE_ME'

pd.concat([
    result[result['word'] == word].assign(comparison='all speeches'),
    result_government[result_government['word'] == word].assign(comparison='in government'),
    result_opposition[result_opposition['word'] == word].assign(comparison='in opposition'),
])[['comparison', 'word', 'fpm_Conservative', 'fpm_Labour', 'log_ratio', 'significant']]

### Women and men

The same problem comes up with gender. Women made very few of the speeches in the 1980s, and many more recently, so anything that changed over time will look "female".

In [ ]:
#@title Distinctive words: women vs. men (all years)
result_gender = p4p.distinctive_words_by_group(df, 'gender', 'female', 'male', min_texts=20)
p4p.plot_distinctive_words(result_gender, by='g2')

∇ *Which of these words might be about gender, and which about period?*

In [ ]:
#@title TODO: Women vs. men, only since 2010
df_since_2010 = df[df['year'] >= REPLACE_ME]

result_gender_since_2010 = p4p.distinctive_words_by_group(df_since_2010, 'gender', 'female', 'male', min_texts=20)
p4p.plot_distinctive_words(result_gender_since_2010, by='g2')

∇ *What's left? Look at the men's words as well as the women's: are there differences in **topic**, in **style** (how people argue), or both?*

∇ *Women MPs since 2010 are more often Labour than Conservative. How could you check whether these words are about gender or about party? (Hint: compare women and men within one party.)*

In [ ]:
#@title TODO: Women vs. men, within one party, since 2010

### Frontbench and backbench

`frontbench` is `True` or `False`, which would make confusing labels on a plot. Let's make a clearer column, and (as we saw in Part 1) only use speeches since 1997.

In [ ]:
#@title Distinctive words: frontbench vs. backbench, since 1997
df['bench'] = df['frontbench'].map({True: 'frontbench', False: 'backbench'})
df_since_1997 = df[df['year'] >= 1997]

result_bench = p4p.distinctive_words_by_group(df_since_1997, 'bench', 'frontbench', 'backbench', min_texts=20)
p4p.plot_distinctive_words(result_bench, by='g2')

∇ *What does a minister (or shadow minister) sound like? What about a backbencher? What are backbenchers asking for, and on whose behalf?*

In [ ]:
#@title TODO: Read a backbench word in context (try "page": why is it there?)
p4p.kwic(df_since_1997[df_since_1997['bench'] == 'backbench'], 'REPLACE_ME').sample(20, random_state=1)

### Words over time

`p4p.plot_word_over_time` draws one point per year: the word's frequency in all of that year's speeches together (or, with `color=`, in each group's speeches that year).

In [ ]:
#@title "Europe" and "European", by party
p4p.plot_word_over_time(df, ['europe', 'european'], color='party_main')

In [ ]:
#@title TODO: Plot a distinctive word from your results over time, colored by party or side
p4p.plot_word_over_time(df, 'REPLACE_ME', color='REPLACE_ME_ALSO')

In [ ]:
#@title TODO: Read the same word in context
p4p.kwic(df, 'REPLACE_ME').sample(20, random_state=1)

∇ *Does reading the word in context confirm the story you told about it, or complicate it?*

## Your turn

In [ ]:
#@title Compare the 1980s and the 2010s
df_80s_and_10s = df[df['decade'].isin([1980, 2010])]

result_decades = p4p.distinctive_words_by_group(df_80s_and_10s, 'decade', 1980, 2010, min_texts=20)
p4p.plot_distinctive_words(result_decades)

∇ *What was Parliament talking about in the 1980s that it isn't now, and vice versa? What kinds of words are these (names, institutions, technologies, ideas)?*

In [ ]:
#@title TODO: Compare two other parties (e.g. 'SNP' and 'Labour', or 'Liberal Democrat' and 'Conservative')
result_parties = p4p.distinctive_words_by_group(df, 'party_main', 'REPLACE_ME', 'REPLACE_ME_ALSO', min_texts=20)
p4p.plot_distinctive_words(result_parties)

In [ ]:
#@title TODO: Compare two speakers (check the `speaker` column for exact names)
result_speakers = p4p.distinctive_words_by_group(df, 'speaker', 'REPLACE_ME', 'REPLACE_ME_ALSO', min_texts=5)
p4p.plot_distinctive_words(result_speakers)

∇ *For your two speakers, what else differs besides who they are (party, position, period, gender)? Which of their distinctive words might really be about one of those?*

In [ ]:
#@title TODO: Pick one of your comparisons. Sort and filter the table, plot a word over time, read it in context, and write a few sentences interpreting what you found